In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
# Question 1 -- version of Spark
spark = SparkSession.builder.master("local[1]") \
                    .appName('test-spark') \
                    .getOrCreate()


print(f'The PySpark {spark.version} version is running...')

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/05 23:57:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


The PySpark 3.5.4 version is running...


In [3]:
# Question 2 -- approx size of parquet files created by dividing original into 4 partitions
yt_202410_df = spark.read.parquet('yellow_tripdata_2024-10.parquet')

yt_202410_df.repartition(4).write.mode('overwrite').parquet('yellow_tripdata_2024-10_repart.parquet')

!ls -l yellow_tripdata_2024-10_repart.parquet

[Stage 3:============================================>              (3 + 1) / 4]

total 197120
-rw-r--r--  1 dnadler  staff         0 Mar  5 23:57 _SUCCESS
-rw-r--r--  1 dnadler  staff  24912507 Mar  5 23:57 part-00000-69a0d507-f8e8-490a-b4aa-446e90fef962-c000.snappy.parquet
-rw-r--r--  1 dnadler  staff  24947918 Mar  5 23:57 part-00001-69a0d507-f8e8-490a-b4aa-446e90fef962-c000.snappy.parquet
-rw-r--r--  1 dnadler  staff  24933728 Mar  5 23:57 part-00002-69a0d507-f8e8-490a-b4aa-446e90fef962-c000.snappy.parquet
-rw-r--r--  1 dnadler  staff  24962053 Mar  5 23:57 part-00003-69a0d507-f8e8-490a-b4aa-446e90fef962-c000.snappy.parquet


In [4]:
yt_202410_df.createOrReplaceTempView('yt_202410_data')

In [5]:
 yt_202410_sample = spark.sql("SELECT * FROM yt_202410_data LIMIT 5")

yt_202410_sample.toPandas()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,2,2024-10-01 00:30:44,2024-10-01 00:48:26,1,3.0,1,N,162,246,1,18.4,1.0,0.5,1.5,0.0,1.0,24.9,2.5,0.0
1,1,2024-10-01 00:12:20,2024-10-01 00:25:25,1,2.2,1,N,48,236,1,14.2,3.5,0.5,3.8,0.0,1.0,23.0,2.5,0.0
2,1,2024-10-01 00:04:46,2024-10-01 00:13:52,1,2.7,1,N,142,24,1,13.5,3.5,0.5,3.7,0.0,1.0,22.2,2.5,0.0
3,1,2024-10-01 00:12:10,2024-10-01 00:23:01,1,3.1,1,N,233,75,1,14.2,3.5,0.5,2.0,0.0,1.0,21.2,2.5,0.0
4,1,2024-10-01 00:30:22,2024-10-01 00:30:39,1,0.0,1,N,262,262,3,3.0,3.5,0.5,0.0,0.0,1.0,8.0,2.5,0.0


In [6]:
# Question 3
#Naive, no filtering for "valid ride",e.g. trip_distance > 0 and/or fare_amount > 0
record_count1 = spark.sql(
    "SELECT COUNT(*) FROM yt_202410_data WHERE DATE(tpep_pickup_datetime) = '2024-10-15'"   
)
print("Without filtering...")
display(record_count1.toPandas())

# Filter for fare_amount > 0
record_count2 = spark.sql(
    "SELECT COUNT(*) FROM yt_202410_data WHERE DATE(tpep_pickup_datetime) = '2024-10-15'"
    " AND fare_amount > 0"
)
print("Filter for fare_amount > 0")
display(record_count2.toPandas())

# Filter for trip_distance > 0
record_count3 = spark.sql(
    "SELECT COUNT(*) FROM yt_202410_data WHERE DATE(tpep_pickup_datetime) = '2024-10-15'"
    " AND trip_distance > 0"
)
print("Filter for trip_distance > 0")
display(record_count3.toPandas())

# Filter for both fare_amount > 0 and trip_distance > 0
record_count4 = spark.sql(
    "SELECT COUNT(*) FROM yt_202410_data WHERE DATE(tpep_pickup_datetime) = '2024-10-15'"
    " AND fare_amount > 0 AND trip_distance > 0"
)
print("Filter for both fare_amount > 0 and trip_distance > 0")
display(record_count4.toPandas())

record_count5 = spark.sql(
    "SELECT COUNT(*) FROM yt_202410_data WHERE DATE(tpep_pickup_datetime) = '2024-10-15'"
    " AND DATE(tpep_dropoff_datetime) = '2024-10-15'AND fare_amount > 0 AND trip_distance > 0"
)
print("Filter for both fare_amount > 0 and trip_distance > 0")
print("as well as for dropoff date also October 15th")
record_count5.toPandas()

Without filtering...


,count(1)
0,128893


Filter for fare_amount > 0


,count(1)
0,126164


Filter for trip_distance > 0


,count(1)
0,126106


Filter for both fare_amount > 0 and trip_distance > 0


,count(1)
0,123639


Filter for both fare_amount > 0 and trip_distance > 0
as well as for dropoff date also October 15th


,count(1)
0,122773


In [7]:
# Question 4 -- duration of longest trip in hours
# I've drive across the lower 48 of the U.S. in less time!
longest_trip = spark.sql(
    'SELECT MAX(DATEDIFF(hour, tpep_pickup_datetime, tpep_dropoff_datetime)) FROM yt_202410_data'
)
longest_trip.toPandas()

,"max(timestampdiff(hour, tpep_pickup_datetime, tpep_dropoff_datetime))"
0,162


In [9]:
# Question 5 -- port on which the Spark Web UI runs
# We know that not having chained a call to config('spark.ui.port'=<PORT NUM>), 
# the default is 4040, But we can also get this port number from the context web url.
# https://stackoverflow.com/questions/37923380/determine-spark-ui-port-from-within-jupyter-pyspark
spark.sparkContext.uiWebUrl

'http://10.0.0.178:4040'

In [10]:
zone_df = spark.read \
    .option("header", "true") \
    .csv('taxi_zone_lookup.csv')

zone_df.createOrReplaceTempView('zones')


In [11]:
zones_sample = spark.sql('SELECT * FROM zones LIMIT 5')
zones_sample.toPandas()

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


In [12]:
# QUestion 6
least_popular_pickup_zones = spark.sql(
    'SELECT trips.PULocationID, ANY_VALUE(zones.Zone), COUNT(*) as count'
    ' FROM yt_202410_data as trips INNER JOIN zones'
    ' ON trips.PULocationID = zones.LocationID'
    ' GROUP BY PULocationID ORDER BY count'
)
least_popular_pickup_zones.toPandas()

,PULocationID,any_value(Zone),count
0,105,Governor's Island/Ellis Island/Liberty Island,1
1,5,Arden Heights,2
2,199,Rikers Island,2
3,2,Jamaica Bay,3
4,111,Green-Wood Cemetery,3
...,...,...,...
256,162,Midtown East,132055
257,236,Upper East Side North,167231
258,161,Midtown Center,177568
259,237,Upper East Side South,191011
